# 数值稳定性（Numerical Stability）

对应课程：`phases/01-math-foundations/13-numerical-stability`

> 浮点数是一个有漏洞的抽象。公式写对了，训练照样可能变成 `inf` / `NaN`。

本 notebook 把 `numerical.py` 里的函数拆开：每个函数一组中文注释，后面跟一小段可运行实验。完整打印型 demo 仍在 `numerical.py`。

**贯穿全课的模式：** naive = 数学直译；stable = 改计算顺序，让浮点不溢出、不抵消、不 `log(0)`。


## 0. 依赖

只用标准库，和课程允许清单一致。


In [1]:
import math
import struct
import random


## 1. Softmax：把 logits 变成概率

$$
\mathrm{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

`exp(100)` 在 float32 里已经是 `inf`。稳定写法先减最大值：分子分母同乘 $e^{-c}$，概率不变，最大项变成 `exp(0)=1`。


In [2]:
def softmax_naive(logits):
    """按公式直接 exp，再归一化。logits 稍大就会 overflow。"""
    exps = [math.exp(z) for z in logits]
    total = sum(exps)
    return [e / total for e in exps]


def softmax_stable(logits):
    """减 max 后再 exp。数学等价，数值安全。"""
    max_logit = max(logits)
    # z - max 全 <= 0，所以 exp(...) 落在 (0, 1]，不会炸
    exps = [math.exp(z - max_logit) for z in logits]
    total = sum(exps)
    return [e / total for e in exps]


safe = [2.0, 1.0, 0.1]
print("安全输入:", safe)
print("naive :", [round(p, 6) for p in softmax_naive(safe)])
print("stable:", [round(p, 6) for p in softmax_stable(safe)])

# Python 的 math.exp 是 float64，约 exp(709) 才 overflow。
# 训练里常用 float32，大约 exp(88) 就会变成 inf。
dangerous = [800.0, 801.0, 802.0]
print("\n危险输入 (float64 才会炸):", dangerous)
print("stable:", [round(p, 6) for p in softmax_stable(dangerous)])
try:
    print("naive :", softmax_naive(dangerous))
except OverflowError:
    print("naive : OverflowError  (exp(802) 超出 float64)")


安全输入: [2.0, 1.0, 0.1]
naive : [0.659001, 0.242433, 0.098566]
stable: [0.659001, 0.242433, 0.098566]

危险输入 (float64 才会炸): [800.0, 801.0, 802.0]
stable: [0.090031, 0.244728, 0.665241]
naive : OverflowError  (exp(802) 超出 float64)


## 2. Log-Sum-Exp：softmax 的对数分母

$$
\mathrm{LSE}(x) = \log\sum_i e^{x_i}
= c + \log\sum_i e^{x_i - c},\quad c=\max(x)
$$

这是 softmax / 交叉熵 / 序列对数概率的公共件。

- 某个 $x_i$ 很大 → `exp` 上溢
- 全体很负 → `exp` 下溢成 0，然后 `log(0) = -inf`

减 max 之后：最大项是 1，和至少是 1，`log` 不会到 `-inf`。


In [3]:
def logsumexp_naive(values):
    """log(sum(exp(x))) 直译。大数 overflow，极小数 log(0)。"""
    return math.log(sum(math.exp(v) for v in values))


def logsumexp_stable(values):
    """Log-Sum-Exp Trick：先减 max，再 log-sum-exp。"""
    c = max(values)
    # 提出 e^c 之后变成 c + log(sum(exp(x-c)))
    return c + math.log(sum(math.exp(v - c) for v in values))


def log_softmax_stable(logits):
    """log softmax(z_i) = z_i - LSE(z)。全程对数空间，避免 log(0)。"""
    c = max(logits)
    lse = c + math.log(sum(math.exp(z - c) for z in logits))
    return [z - lse for z in logits]


safe = [1.0, 2.0, 3.0]
print("安全值 naive / stable:", logsumexp_naive(safe), logsumexp_stable(safe))

large = [800.0, 801.0, 802.0]
print("大数 stable:", logsumexp_stable(large))
try:
    print("大数 naive :", logsumexp_naive(large))
except OverflowError:
    print("大数 naive : OverflowError")

print("极负 stable:", logsumexp_stable([-1000.0, -999.0, -998.0]))
print("全相等 (应为 5+ln3):", logsumexp_stable([5.0, 5.0, 5.0]), 5.0 + math.log(3.0))
print("log_softmax:", [round(v, 6) for v in log_softmax_stable([2.0, 5.0, 1.0])])


安全值 naive / stable: 3.4076059644443806 3.4076059644443806
大数 stable: 802.4076059644444
大数 naive : OverflowError
极负 stable: -997.5923940355556
全相等 (应为 5+ln3): 6.09861228866811 6.09861228866811
log_softmax: [-3.065884, -0.065884, -4.065884]


## 3. 交叉熵：$-\log p_y$

naive 路径：softmax → 再 `log`。模型错得很自信时 $p_y\to 0$，`log(0)=-inf`。

stable 路径：$-\log p_y = \mathrm{LSE}(z) - z_y$，从不构造接近 0 的概率。


In [4]:
def cross_entropy_naive(true_class, logits):
    """-log(softmax(z)_y)。依赖 naive softmax，大 logits 会炸。"""
    probs = softmax_naive(logits)
    return -math.log(probs[true_class])


def cross_entropy_stable(true_class, logits):
    """-log_softmax(z)_y = LSE(z) - z_y。"""
    log_probs = log_softmax_stable(logits)
    return -log_probs[true_class]


logits = [2.0, 5.0, 1.0]
print("普通 logits  naive/stable:", cross_entropy_naive(1, logits), cross_entropy_stable(1, logits))

large_logits = [800.0, 805.0, 799.0]
print("大 logits stable:", cross_entropy_stable(1, large_logits))
try:
    print("大 logits naive :", cross_entropy_naive(1, large_logits))
except (OverflowError, ValueError):
    print("大 logits naive : OVERFLOW / NaN")

print("自信且对:", cross_entropy_stable(2, [0.0, 0.0, 50.0]))
print("自信但错:", cross_entropy_stable(0, [0.0, 0.0, 50.0]))


普通 logits  naive/stable: 0.06588390375742913 0.06588390375742925
大 logits stable: 0.009174484591767396
大 logits naive : OVERFLOW / NaN
自信且对: -0.0
自信但错: 50.0


## 4. Sigmoid 与二元交叉熵

naive sigmoid $1/(1+e^{-x})$：`x` 很负时 `exp(-x)` overflow。

stable：永远只对 **非正数** 做 `exp`。

- $x\ge 0$：用 $1/(1+e^{-x})$
- $x< 0$：改写成 $e^{x}/(1+e^{x})$

BCE naive 吃概率 $p$，两端 `log(0)`。stable 吃 **logit**，对应 `binary_cross_entropy_with_logits`：

$$
\max(z,0)+\log\bigl(e^{-\max(z,0)}+e^{z-\max(z,0)}\bigr)-yz
$$


In [5]:
def sigmoid_naive(x):
    """1 / (1 + exp(-x))。x 很负时 exp(-x) overflow。"""
    return 1.0 / (1.0 + math.exp(-x))


def sigmoid_stable(x):
    """按符号分支，exp 的参数始终 <= 0。"""
    if x >= 0:
        z = math.exp(-x)          # -x <= 0
        return 1.0 / (1.0 + z)
    z = math.exp(x)               # x < 0
    return z / (1.0 + z)


def binary_cross_entropy_naive(y_true, y_pred):
    """-(y log p + (1-y) log(1-p))。p=0 或 1 时 log 炸。"""
    return -(y_true * math.log(y_pred) + (1 - y_true) * math.log(1 - y_pred))


def binary_cross_entropy_stable(y_true, logit):
    """从 logit 直接算 BCE，等价于 log(1+e^z) - yz 的稳定形式。"""
    max_val = max(0.0, logit)  # 提出正部，避免 exp(大正数)
    return max_val + math.log(math.exp(-max_val) + math.exp(logit - max_val)) - y_true * logit


print(f"{'x':>8}  {'naive':>14}  {'stable':>14}")
for x in [0.0, 10.0, -10.0, 100.0, -100.0, 710.0, -710.0]:
    try:
        naive = f"{sigmoid_naive(x):.10f}"
    except OverflowError:
        naive = "OVERFLOW"
    print(f"{x:>8.1f}  {naive:>14}  {sigmoid_stable(x):.10f}")

print("\nBCE(y=1, logit=3) stable:", binary_cross_entropy_stable(1.0, 3.0))
print("BCE(y=0, logit=3) stable:", binary_cross_entropy_stable(0.0, 3.0))


       x           naive          stable
     0.0    0.5000000000  0.5000000000
    10.0    0.9999546021  0.9999546021
   -10.0    0.0000453979  0.0000453979
   100.0    1.0000000000  1.0000000000
  -100.0    0.0000000000  0.0000000000
   710.0    1.0000000000  1.0000000000
  -710.0        OVERFLOW  0.0000000000

BCE(y=1, logit=3) stable: 0.048587351573742055
BCE(y=0, logit=3) stable: 3.048587351573742


## 5. 梯度检查：用数值梯度验收解析梯度

中心差分：

$$
\frac{\partial f}{\partial x_i}\approx\frac{f(x+he_i)-f(x-he_i)}{2h}
$$

`h` 太小会灾难性抵消，默认 `1e-5`。相对误差 $<10^{-5}$ 通常认为解析梯度写对了。


In [6]:
def numerical_gradient(f, x, h=1e-5):
    """对每个分量做中心有限差分。"""
    grad = []
    for i in range(len(x)):
        x_plus = x[:]
        x_minus = x[:]
        x_plus[i] += h
        x_minus[i] -= h
        grad.append((f(x_plus) - f(x_minus)) / (2 * h))
    return grad


def check_gradient(analytical, numerical, tolerance=1e-5):
    """比较解析梯度 vs 数值梯度。返回是否全部通过。"""
    all_ok = True
    for i, (a, n) in enumerate(zip(analytical, numerical)):
        denom = max(abs(a), abs(n), 1e-8)  # 避免除零
        rel_error = abs(a - n) / denom
        status = "OK" if rel_error < tolerance else "FAIL"
        if status == "FAIL":
            all_ok = False
        print(f"  param {i}: analytical={a:.8f} numerical={n:.8f} "
              f"rel_error={rel_error:.2e} [{status}]")
    return all_ok


def f_ce(logits):
    return cross_entropy_stable(0, logits)


logits = [2.0, 1.0, 0.5]
probs = softmax_stable(logits)
# 交叉熵对 logits 的梯度是 p_i - 1[i=y]
analytical = [probs[i] - (1.0 if i == 0 else 0.0) for i in range(len(logits))]
print("交叉熵梯度检查:")
check_gradient(analytical, numerical_gradient(f_ce, logits))


交叉熵梯度检查:
  param 0: analytical=-0.37146828 numerical=-0.37146828 rel_error=3.71e-11 [OK]
  param 1: analytical=0.23122390 numerical=0.23122390 rel_error=6.75e-11 [OK]
  param 2: analytical=0.14024438 numerical=0.14024438 rel_error=5.13e-11 [OK]


## 6. 梯度裁剪：挡住 explosion

- **clip by value**：每个分量夹到 $[-m,m]$，**会改变方向**
- **clip by norm**：整体 $\ell_2$ 超过阈值就等比缩小，**方向不变**

训练里几乎总是用 clip by norm。


In [7]:
def clip_by_value(gradients, max_val):
    """逐元素裁剪。方向会变。"""
    return [max(-max_val, min(max_val, g)) for g in gradients]


def clip_by_norm(gradients, max_norm):
    """按整体 L2 范数缩放。方向不变。"""
    total_norm = math.sqrt(sum(g ** 2 for g in gradients))
    if total_norm > max_norm:
        scale = max_norm / total_norm
        return [g * scale for g in gradients]
    return list(gradients)


grads = [10.0, 20.0, 30.0]
print("raw          :", grads)
print("clip value 15:", clip_by_value(grads, 15.0))
clipped = clip_by_norm(grads, 5.0)
print("clip norm  5 :", [round(g, 4) for g in clipped])
print("clipped L2   :", round(math.sqrt(sum(g ** 2 for g in clipped)), 4))


raw          : [10.0, 20.0, 30.0]
clip value 15: [10.0, 15.0, 15.0]
clip norm  5 : [1.3363, 2.6726, 4.0089]
clipped L2   : 5.0


## 7. NaN / Inf 检测

一个 `nan` 会污染 sum、mean、后续所有层。训练 loss 变 NaN 时，先扫 weights / logits / grads。


In [8]:
def check_tensor(name, values):
    """统计 NaN / Inf。全部有限返回 True。"""
    n_nan = sum(1 for v in values if math.isnan(v))
    n_inf = sum(1 for v in values if math.isinf(v))
    if n_nan or n_inf:
        print(f"  WARNING {name}: {n_nan} NaN, {n_inf} Inf / {len(values)}")
        return False
    print(f"  OK {name}: all {len(values)} finite")
    return True


check_tensor("weights", [0.1, -0.3, 0.5])
check_tensor("logits_bad", [1.0, float("inf"), -2.0])
check_tensor("grads_bad", [0.01, float("nan"), -0.03])
print("nan 会传染: sum([1, nan, 3]) =", sum([1.0, float("nan"), 3.0]))


  OK weights: all 3 finite
  WARNING logits_bad: 0 NaN, 1 Inf / 3
  WARNING grads_bad: 1 NaN, 0 Inf / 3
nan 会传染: sum([1, nan, 3]) = nan


## 8. 模拟 float16 / bfloat16

Python 默认 float64。这里人为丢掉精度，看混合精度会在哪死。

| 格式 | 指数位 | 范围 | 训练含义 |
|------|--------|------|----------|
| float16 | 5 | 最大约 65504 | 激活/梯度容易 overflow 或 underflow 成 0 |
| bfloat16 | 8（同 float32） | 到 3.4e38 | 范围优先，训练更常用 |


In [9]:
def simulate_bfloat16(x):
    """float32 砍掉低 16 位尾数，留下 bf16 的位型。"""
    packed = struct.pack("f", x)
    as_int = int.from_bytes(packed, "little")
    truncated = as_int & 0xFFFF0000  # 高 16 位 = 1 sign + 8 exp + 7 mantissa
    return struct.unpack("f", truncated.to_bytes(4, "little"))[0]


def simulate_float16(x):
    """IEEE float16。超出范围变成 inf。"""
    try:
        packed = struct.pack("e", x)
        return struct.unpack("e", packed)[0]
    except (OverflowError, struct.error):
        return float("inf") if x > 0 else float("-inf")


print(f"{'value':>12}  {'float16':>12}  {'bfloat16':>12}")
for v in [1.0, 0.1, math.pi, 65504.0, 65536.0, 100000.0]:
    f16 = simulate_float16(v)
    bf16 = simulate_bfloat16(v)
    f16_s = f"{f16:.4f}" if not math.isinf(f16) else "inf"
    print(f"  {v:>10.4f}  {f16_s:>12}  {bf16:12.4f}")


       value       float16      bfloat16
      1.0000        1.0000        1.0000
      0.1000        0.1000        0.0996
      3.1416        3.1406        3.1406
  65504.0000    65504.0000    65280.0000
  65536.0000           inf    65536.0000
  100000.0000           inf    99840.0000


## 9. 稳定求和、方差、LayerNorm

**Kahan**：补偿求和，把「上次没加进去的那一点」记下来再补。

**方差 naive** $\mathbb{E}[x^2]-\mathbb{E}[x]^2$：均值很大时两个大数相减，灾难性抵消。

**Welford**：在线用 $x-\mathrm{mean}$ 更新，避开大平方相减。

**LayerNorm** 的 $\varepsilon$：输入全相等时方差为 0，防止除零；同时也把激活值按层拉回有界范围。


In [10]:
def kahan_sum(values):
    """补偿求和：把每次丢失的低位存进 compensation。"""
    total = 0.0
    compensation = 0.0
    for v in values:
        y = v - compensation      # 补上一次丢掉的部分
        t = total + y
        compensation = (t - total) - y  # 新的舍入误差
        total = t
    return total


def welford_variance(values):
    """在线方差。用增量更新 mean 和 M2，避免 E[x^2]-E[x]^2。"""
    n = 0
    mean = 0.0
    m2 = 0.0
    for x in values:
        n += 1
        delta = x - mean
        mean += delta / n
        delta2 = x - mean
        m2 += delta * delta2
    if n < 2:
        return 0.0
    return m2 / n  # 总体方差（除以 n，不是 n-1）


def variance_naive(values):
    """E[x^2] - E[x]^2。大均值时有效数字对消。"""
    n = len(values)
    mean_x = sum(values) / n
    mean_x2 = sum(v ** 2 for v in values) / n
    return mean_x2 - mean_x ** 2


def layer_norm(values, epsilon=1e-5, gamma=1.0, beta=0.0):
    """(x-mean)/sqrt(var+eps)*gamma + beta。eps 防止除零。"""
    n = len(values)
    mean = sum(values) / n
    var = sum((v - mean) ** 2 for v in values) / n
    std = math.sqrt(var + epsilon)
    return [(v - mean) / std * gamma + beta for v in values]


xs = [1e-7] * 100_000
true = 1e-7 * 100_000
print(f"Kahan vs naive 累加 1e-7 * 1e5: true={true}")
print(f"  naive={sum(xs):.10f}  kahan={kahan_sum(xs):.10f}")

data = [1_000_000.0, 1_000_001.0, 1_000_002.0]
print(f"\n方差 (真值 2/3={2/3:.10f})")
print(f"  naive  : {variance_naive(data):.10f}")
print(f"  welford: {welford_variance(data):.10f}")

print("\n常数输入 + LayerNorm (靠 eps 活下来):", [round(v, 4) for v in layer_norm([5.0, 5.0, 5.0, 5.0])])


Kahan vs naive 累加 1e-7 * 1e5: true=0.01
  naive=0.0100000000  kahan=0.0100000000

方差 (真值 2/3=0.6666666667)
  naive  : 0.6666259766
  welford: 0.6666666667

常数输入 + LayerNorm (靠 eps 活下来): [0.0, 0.0, 0.0, 0.0]


## 10. 损失缩放（loss scaling）直觉

float16 最小正规数大约 `6e-8`。很多梯度比这还小，直接转 f16 会变成 0。

做法：反传前把 loss 乘一个大系数（如 1024），梯度变大、能被 f16 表示；更新权重前再除回去。溢出就把 scale 减半（动态损失缩放）。


In [11]:
random.seed(42)
tiny = [random.uniform(1e-9, 1e-5) for _ in range(1000)]
zeros_raw = sum(1 for g in tiny if simulate_float16(g) == 0.0)
scale = 1024.0
zeros_scaled = sum(1 for g in tiny if simulate_float16(g * scale) == 0.0)
print(f"无缩放后变 0: {zeros_raw}/1000")
print(f"x{scale:.0f} 后变 0: {zeros_scaled}/1000")


无缩放后变 0: 4/1000
x1024 后变 0: 0/1000


## 对照表

| 函数 | 角色 |
|------|------|
| `softmax_naive` / `softmax_stable` | 概率归一化；stable 减 max |
| `logsumexp_naive` / `logsumexp_stable` | $\log\sum e^{x}$；课文核心 trick |
| `log_softmax_stable` | $z_i - \mathrm{LSE}(z)$ |
| `cross_entropy_*` | $-\log p_y$；stable 不经过概率 |
| `sigmoid_*` / `binary_cross_entropy_*` | 二分类同样的 overflow 问题 |
| `numerical_gradient` / `check_gradient` | 验收手写梯度 |
| `clip_by_value` / `clip_by_norm` | 挡梯度爆炸 |
| `check_tensor` | NaN/Inf 扫描 |
| `simulate_float16` / `simulate_bfloat16` | 看低精度会在哪死 |
| `kahan_sum` / `welford_variance` / `variance_naive` | 累加与灾难性抵消 |
| `layer_norm` | $\varepsilon$ + 按层把激活拉回有界 |

要看完整 14 段打印 demo，运行：

```bash
python numerical.py
```
